# Transfer Learning — Aula 1: Conhecendo uma rede pré-treinada

## A ideia
Treinar uma CNN do zero (como no artigo) obriga a rede a reaprender até o básico — o que é uma borda, uma textura — a partir de apenas ~15 mil imagens. É um desperdício, e é parte do motivo de o modelo empacar em ~98%.

O **transfer learning** reaproveita uma rede já treinada em um dataset gigante (o **ImageNet**: 1,4 milhão de imagens). As camadas iniciais dessa rede já sabem enxergar bordas, formas e texturas — conhecimento que serve para qualquer imagem, raio-X incluso. Aproveitamos essa parte pronta e treinamos apenas uma "cabeça" nova para as nossas 4 classes, usando a nossa base de dados.

## Nesta aula
Antes de montar qualquer coisa, vamos apenas **carregar e inspecionar** a rede pré-treinada MobileNetV2, para enxergar na prática o que ela é: quantas camadas tem, quantos parâmetros, e qual é a saída original dela.

## 1. Carregando a MobileNetV2 pré-treinada

Usamos `keras.applications.MobileNetV2` para baixar a rede já treinada no ImageNet:
- `weights="imagenet"` → traz os pesos aprendidos nas 1,4 milhão de imagens (o "olho treinado").
- `include_top=True` → por enquanto, inclui a **cabeça original** da rede, aquela que classifica nas 1.000 categorias do ImageNet (gato, carro, cadeira...).

Na primeira vez, o Keras baixa os pesos da internet (~14 MB) e guarda no computador; nas próximas, usa a cópia local.

In [1]:
import tensorflow as tf
from tensorflow import keras

# Baixa a MobileNetV2 já treinada no ImageNet (na 1ª vez, baixa os pesos ~14 MB)
modelo_pretreinado = keras.applications.MobileNetV2(
    weights="imagenet",   # pesos aprendidos no ImageNet
    include_top=True      # inclui a cabeça original (1000 classes do ImageNet)
)

print("Número de camadas:", len(modelo_pretreinado.layers))
print("Total de parâmetros:", f"{modelo_pretreinado.count_params():,}")
print("Formato da saída:", modelo_pretreinado.output_shape)

I0000 00:00:1791558162.346067     794 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791558162.773253     794 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791558164.741406     794 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1791558166.550917     794 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 10s 1us/step
Número de camadas: 156
Total de parâmetros: 3,538,984
Formato da saída: (None, 1000)


## 2. Carregando a base pré-treinada (sem a cabeça) e congelando

- `include_top=False` → traz apenas o extrator de características, sem a cabeça de 1000 classes do ImageNet.
- `input_shape=(224, 224, 3)` → tamanho de entrada padrão da MobileNetV2 (ela foi treinada em imagens 224×224). Vamos redimensionar nossos raios-X para esse tamanho mais adiante.
- `base.trainable = False` → **congela** a base: seus pesos (aprendidos no ImageNet) não serão alterados durante o treino, preservando o conhecimento pré-treinado. Apenas a cabeça nova (criada depois) será treinada.

In [2]:
import numpy as np

# Carrega SÓ o extrator de características (sem a cabeça de 1000 classes)
base = keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Conta os parâmetros treináveis ANTES de congelar
treinaveis_antes = int(sum(np.prod(w.shape) for w in base.trainable_weights))

# CONGELA a base
base.trainable = False

# Conta de novo DEPOIS de congelar
treinaveis_depois = int(sum(np.prod(w.shape) for w in base.trainable_weights))

print("Formato da saída da base (sem cabeça):", base.output_shape)
print(f"Parâmetros treináveis ANTES de congelar:  {treinaveis_antes:,}")
print(f"Parâmetros treináveis DEPOIS de congelar: {treinaveis_depois:,}")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 7s 1us/step
Formato da saída da base (sem cabeça): (None, 7, 7, 1280)
Parâmetros treináveis ANTES de congelar:  2,223,872
Parâmetros treináveis DEPOIS de congelar: 0


## 3. O preprocessamento próprio da MobileNetV2

Cada rede pré-treinada espera a entrada na mesma faixa de valores em que foi treinada. A MobileNetV2 **não** usa pixels de 0 a 1 (como o nosso `Rescaling(1./255)`), e sim de **−1 a +1**.

Por isso, em vez do `Rescaling`, usamos a função `preprocess_input` da própria MobileNetV2, que converte os pixels de 0–255 para a faixa −1 a +1. Abaixo, um exemplo mostra essa conversão.